In [1]:
using Pkg
Pkg.add("DifferentialEquations")
Pkg.add("DelayDiffEq")
Pkg.add("Plots")
Pkg.add("LsqFit")
Pkg.add("ComplexityMeasures")
Pkg.add("Hurst")
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("StatsBase")
Pkg.add("TransferEntropy")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages ad

## Full simulation

In [13]:
using DifferentialEquations, DelayDiffEq
using Plots, Random, Distributions
using ComplexityMeasures
using Hurst
using Statistics
using LsqFit
using StatsBase
using TransferEntropy: VisitationFrequency, RectangularBinning, transferentropy
using CSV, DataFrames

# Translation variables
@inline function pq(ϕ, c)
    imax = length(ϕ)
    p = zeros(imax)
    q = zeros(imax)
    p[1] = ϕ[1] * cos(c)
    q[1] = ϕ[1] * sin(c)
    for i in 2:imax
        p[i] = p[i-1] + ϕ[i-1]*cos(c * (i-1))
        q[i] = q[i-1] + ϕ[i-1]*sin(c * (i-1))
    end
    return p, q
end

# Mean square displacement
@inline function Mn_c(ϕ, c, ncut)
    p, q = pq(ϕ, c)
    N = length(ϕ) - ncut
    Mn = zeros(ncut)
    for n in 1:ncut
        Mn[n] = mean([(p[j+n] - p[j])^2 + (q[j+n] - q[j])^2 for j in 1:N])
    end
    return Mn
end

# Oscillatory term
function Vosc_c(ϕ, c, ncut)
    Eϕ = mean(ϕ)
    return [Eϕ^2 * (1 - cos(n*c))/(1 - cos(c)) for n in 1:ncut]
end

function Dn_c(ϕ, c, ncut)
    return Mn_c(ϕ, c, ncut) - Vosc_c(ϕ, c, ncut)
end

function Dn_c_tilde(ϕ, c, ncut)
    Dn = Dn_c(ϕ, c, ncut)
    return Dn .- min(Dn...)
end

# The asymptotic growth rate
function Kc(ϕ, c, ncut)
    linear_func = (x, p) -> p[1] .+ p[2] .* x
    fit = curve_fit(linear_func, log.(1:ncut), log.(Dn_c_tilde(ϕ, c, ncut)[1:end] .+ 1e-2), [0, 0.5])
    return fit.param[2]
end

@inline function regressionmethod(ϕ, ncut)
    c_range = 0.01:0.05:2π
    return median([Kc(ϕ, c, ncut) for c in c_range])
end

@inline function correlationmethod(ϕ, ncut)
    ξ = 1:ncut;
    c_range = 0.01:0.05:2π
    K_c = [cor(ξ, Dn_c(ϕ, c, ncut)) for c in c_range]
    return median(K_c[.!isnan.(K_c)])
end

# --------------------
# Model parameters
# --------------------
const A = 0.0041
const α = 5.276
const γ = 0.315
const ϵ = 0.0001

const θ = -5    # put between -5, 5: -5, -1, -0.1, -.01, 0.01, .1, 1, 5
const τ = 10     # delay (0, 100) 10, 40, 70, 100

# const θ = -1
# const τ = 10    

# const θ = -0.1
# const τ = 10 

# const θ = -0.01
# const τ = 10 

# const θ = 0.01
# const τ = 10 

# const θ = 0.1
# const τ = 10 

# const θ = 1
# const τ = 10 

# const θ = 5
# const τ = 10 

# const θ = -5
# const τ = 40 

# const θ = -1
# const τ = 40 

# const θ = -0.1
# const τ = 40 

# const θ = -0.01
# const τ = 40 

# const θ = 0.01
# const τ = 40 

# const θ = 0.1
# const τ = 40 

# const θ = 1
# const τ = 40 

# const θ = 5
# const τ = 40 

# const θ = -5
# const τ = 70 

# const θ = -1
# const τ = 70 

# const θ = -0.1
# const τ = 70 

# const θ = -0.01
# const τ = 70 

# const θ = 0.01
# const τ = 70 

# const θ = 0.1
# const τ = 70 

# const θ = 1
# const τ = 70

# const θ = 5
# const τ = 70

# const θ = -5
# const τ = 100

# const θ = -1
# const τ = 100

# const θ = -0.1
# const τ = 100

# const θ = -0.01
# const τ = 100

# const θ = 0.01
# const τ = 100

# const θ = 0.1
# const τ = 100

# const θ = 1
# const τ = 100

# const θ = 5
# const τ = 100

# slow adaptation function
F(x) = (1/60)*(1 + tanh((0.05 - x)/0.001))

# --------------------
# Randomized initial conditions
# --------------------
Random.seed!(1234)

x1_0 = rand(Uniform(-0.1, 0.1))
x2_0 = rand(Uniform(-0.1, 0.1))

u_init = [
    x1_0, 0.1, 0.019,
    x2_0, 0.1, 0.022
]

println("Initial x1 = $x1_0, x2 = $x2_0")

# history for t <= 0
function history(p, t)
    return u_init .+ zero(t)   # continuous constant history
end

# --------------------
# DDE RHS
# --------------------
function coupled_dML_delay!(du, u, h, p, t)
    x1, y1, I1, x2, y2, I2 = u

    # delayed values
    x2τ = h(p, t - p.τ)[4]
    x1τ = h(p, t - p.τ)[1]

    # neuron 1
    du[1] = x1^2*(1 - x1) - y1 + I1 + p.θ*(x2τ - x1)
    du[2] = A*exp(α*x1) - γ*y1
    du[3] = ϵ*(F(x1) - I1)

    # neuron 2
    du[4] = x2^2*(1 - x2) - y2 + I2 + p.θ*(x1τ - x2)
    du[5] = A*exp(α*x2) - γ*y2
    du[6] = ϵ*(F(x2) - I2)
end

# --------------------
# Pack parameters
# --------------------
p = (θ=θ, τ=τ)

# time span
tspan = (0.0, 8000.0)

# set up DDE problem
prob = DDEProblem(
    coupled_dML_delay!,
    u_init,
    history,
    tspan,
    p;
    constant_lags=[τ]
)

# solve
sol = solve(
    prob,
    MethodOfSteps(Tsit5()),
    dtmax=0.1,
    reltol=1e-8,
    abstol=1e-10
)



# Times
t = sol.t

# Extract each component into its own vector
x1 = [u[1] for u in sol.u]
y1 = [u[2] for u in sol.u]
I1 = [u[3] for u in sol.u]

x2 = [u[4] for u in sol.u]
y2 = [u[5] for u in sol.u]
I2 = [u[6] for u in sol.u]

# save to time series
df = DataFrame(
    t = t,
    x1 = x1,
    y1 = y1,
    I1 = I1,
    x2 = x2,
    y2 = y2,
    I2 = I2
)

CSV.write("full_timeseries_theta_$(θ)_tau_$(τ).csv", df)
println("Saved timeseries")

## computing metrics:

tcut = 4000.0
idxs = findall(t .>= tcut)

ds = 10  # or even 10
x1_clean = x1[idxs][1:ds:end]
x2_clean = x2[idxs][1:ds:end]

# Number of steps to compute mean square displacement
ncut = 300               # you can adjust

K_regression_x1 = regressionmethod(x1_clean, ncut)
K_regression_x2 = regressionmethod(x2_clean, ncut)

println("done K")


m = 2
τ_embed = 1

se1 = complexity_normalized(SampleEntropy(m = m, r = 0.2 * std(x1_clean)), x1_clean)
se2 = complexity_normalized(SampleEntropy(m = m, r = 0.2 * std(x2_clean)), x2_clean)

println("done se")

ap1 = complexity(ApproximateEntropy(m = m, r = 0.2 * std(x1_clean)), x1_clean)
ap2 = complexity(ApproximateEntropy(m = m, r = 0.2 * std(x2_clean)), x2_clean)

println("done ap")

we1 = entropy_wavelet(x1_clean)
we2 = entropy_wavelet(x2_clean)

println("done we")

h1 = hurst_exponent(x1_clean, 1:10)
h2 = hurst_exponent(x2_clean, 1:10)

println("done h")

NN = length(x1_clean)
maxlag = min(200, NN÷10)   # robust, scale-free
lags = -maxlag:maxlag

# Compute the cross-correlation
cc = crosscor(x1_clean, x2_clean, lags; demean=true)
println("done cc")

# transfer entropy
est = VisitationFrequency(RectangularBinning(5))

TE_x1_to_x2 = transferentropy(x1_clean, x2_clean, est; base=2)

# Transfer entropy from x2 -> x1
TE_x2_to_x1 = transferentropy(x2_clean, x1_clean, est; base=2)

println("TE(x1 -> x2) = ", TE_x1_to_x2)
println("TE(x2 -> x1) = ", TE_x2_to_x1)

# ε1 = 0.1 * std(vcat(x1_clean, x2_clean))  # simple scale

# CR = CrossRecurrenceMatrix(x1_clean, x2_clean, RecurrenceThreshold(ε1);
#                              metric=Euclidean())
# rqa_results = rqa(CR)

# println("done recurrence")

# df_metrics = DataFrame(
#     K1 = K_regression_x1,
#     K2 = K_regression_x2,
#     se1 = se1,
#     se2 = se2,
#     ap1 = ap1,
#     ap2 = ap2,
#     we1 = we1,
#     we2 = we2,
#     h1  = h1[1],
#     h2  = h2[1],
#     RR  = rqa_results[:RR],
#     DET = rqa_results[:DET],
#     L   = rqa_results[:L],
#     Lmax= rqa_results[:Lmax],
#     LAM = rqa_results[:LAM],
#     ENTR= rqa_results[:ENTR]
# )

df_metrics = DataFrame(
    K1 = K_regression_x1,
    K2 = K_regression_x2,
    se1 = se1,
    se2 = se2,
    ap1 = ap1,
    ap2 = ap2,
    we1 = we1,
    we2 = we2,
    h1  = h1[1],
    h2  = h2[1],
    ccMax = maximum(cc),
    ccMin = minimum(cc),
    T12 = TE_x1_to_x2,
    T21 = TE_x2_to_x1
)

CSV.write("metrics_results_theta_$(θ)_tau_$(τ).csv", df_metrics)
println("Saved metrics")

Initial x1 = -0.03480465422728103, x2 = 0.009810227263113383
Saved timeseries
done K
done se
done ap
done we
done h
done cc
TE(x1 -> x2) = 0.00023519244994263389
TE(x2 -> x1) = 0.0004787492478968147
Saved metrics


## parameter sweep

In [158]:
thetas = range(-5.0, 5.0; length=100)
taus   = range(10, 100; length=100)

nt, ntau = length(thetas), length(taus)

KK  = fill(NaN, nt, ntau)
SE  = fill(NaN, nt, ntau)
AP  = fill(NaN, nt, ntau)
WE  = fill(NaN, nt, ntau)
H   = fill(NaN, nt, ntau)
CMAX = fill(NaN, nt, ntau)
CMIN = fill(NaN, nt, ntau)
TE12Mat = fill(NaN, nt, ntau)
TE21Mat = fill(NaN, nt, ntau)

Random.seed!(1234)

const TE_EST = VisitationFrequency(RectangularBinning(5))
const MAXLAG = 200

count = 0
for (iθ, θ) in enumerate(thetas)
    
    for (iτ, τ) in enumerate(taus)
        
        
        # slow adaptation function
        F(x) = (1/60)*(1 + tanh((0.05 - x)/0.001))
        
        x1_0 = rand(Uniform(-0.1, 0.1))
        x2_0 = rand(Uniform(-0.1, 0.1))
        
        u_init = [
            x1_0, 0.1, 0.019,
            x2_0, 0.1, 0.022
        ]

        if count%100 == 0
            println("θ = $θ, τ = $τ")
            println("Initial x1 = $x1_0, x2 = $x2_0")
            println(" ")
        end

        
        # history for t <= 0
        function history(p, t)
            return u_init .+ zero(t)   # continuous constant history
        end

        function coupled_dML_delay!(du, u, h, p, t)
            x1, y1, I1, x2, y2, I2 = u
        
            # delayed values
            x2τ = h(p, t - p.τ)[4]
            x1τ = h(p, t - p.τ)[1]
        
            # neuron 1
            du[1] = x1^2*(1 - x1) - y1 + I1 + p.θ*(x2τ - x1)
            du[2] = A*exp(α*x1) - γ*y1
            du[3] = ϵ*(F(x1) - I1)
        
            # neuron 2
            du[4] = x2^2*(1 - x2) - y2 + I2 + p.θ*(x1τ - x2)
            du[5] = A*exp(α*x2) - γ*y2
            du[6] = ϵ*(F(x2) - I2)
        end
        
        # --------------------
        # Pack parameters
        # --------------------
        p = (θ=θ, τ=τ)

        # time span
        tspan = (0.0, 8000.0)
        
        # set up DDE problem
        prob = DDEProblem(
            coupled_dML_delay!,
            u_init,
            history,
            tspan,
            p;
            constant_lags=[τ]
        )
        
        # solve
        sol = solve(
            prob,
            MethodOfSteps(Tsit5()),
            dtmax=0.1,
            reltol=1e-8,
            abstol=1e-10
        )
        
        
        
        # Times
        t = sol.t
        
        # Extract each component into its own vector
        x1 = [u[1] for u in sol.u]
        y1 = [u[2] for u in sol.u]
        I1 = [u[3] for u in sol.u]
        
        x2 = [u[4] for u in sol.u]
        y2 = [u[5] for u in sol.u]
        I2 = [u[6] for u in sol.u]

        tcut = 4000.0
        idxs = findall(t .>= tcut)
        
        ds = 10  # or even 10
        x1_clean = x1[idxs][1:ds:end]
        x2_clean = x2[idxs][1:ds:end]
        
        # Number of steps to compute mean square displacement
        ncut = 300              
        
        K_regression_x1 = regressionmethod(x1_clean, ncut)
        K_regression_x2 = regressionmethod(x2_clean, ncut)
        
        m = 2
        τ_embed = 1
        
        se1 = complexity_normalized(SampleEntropy(m = m, r = 0.2 * std(x1_clean)), x1_clean)
        se2 = complexity_normalized(SampleEntropy(m = m, r = 0.2 * std(x2_clean)), x2_clean)
        
        # println("done se")
        
        ap1 = complexity(ApproximateEntropy(m = m, r = 0.2 * std(x1_clean)), x1_clean)
        ap2 = complexity(ApproximateEntropy(m = m, r = 0.2 * std(x2_clean)), x2_clean)
        
        # println("done ap")
        
        we1 = entropy_wavelet(x1_clean)
        we2 = entropy_wavelet(x2_clean)
        
        # println("done we")
        
        h1 = hurst_exponent(x1_clean, 1:10)
        h2 = hurst_exponent(x2_clean, 1:10)
        
        # println("done h")

        # Compute the cross-correlation
        NN = length(x1_clean)
        maxlag = min(MAXLAG, NN÷10)   # robust, scale-free
        lags = -maxlag:maxlag
        
        cc = crosscor(x1_clean, x2_clean, lags; demean=true)
        # println("done cc")

        # # transfer entropy
        # est = VisitationFrequency(RectangularBinning(5))
        
        # Transfer entropy from x1 -> x2
        TE_x1_to_x2 = transferentropy(x1_clean, x2_clean, TE_EST; base=2)
        
        # Transfer entropy from x2 -> x1
        TE_x2_to_x1 = transferentropy(x2_clean, x1_clean, TE_EST; base=2)
        
        # println("TE(x1 -> x2) = ", TE_x1_to_x2)
        # println("TE(x2 -> x1) = ", TE_x2_to_x1)
        

        # KK1[iθ,iτ] = K_regression_x1
        # KK2[iθ,iτ] = K_regression_x2

        # SE1[iθ,iτ] = se1
        # SE2[iθ,iτ] = se2

        # AP1[iθ,iτ] = ap1
        # AP2[iθ,iτ] = ap2

        # WE1[iθ,iτ] = we1
        # WE2[iθ,iτ] = we2

        # H1[iθ,iτ] = h1[1]
        # H2[iθ,iτ] = h2[1]

        KK[iθ,iτ] = (K_regression_x1 + K_regression_x2)/2 
        SE[iθ,iτ] = (se1+se2)/2
        AP[iθ,iτ] = (ap1+ap2)/2
        WE[iθ,iτ] = (we1+we2)/2
        H[iθ,iτ] = (h1[1]+h2[1])/2
        CMAX[iθ,iτ] = maximum(cc)
        CMIN[iθ,iτ] = minimum(cc)
        TE12Mat[iθ,iτ] = TE_x1_to_x2
        TE21Mat[iθ,iτ] = TE_x2_to_x1

        count+=1
                
    end
end

function save_grid(name, M)
    df = DataFrame(M, :auto)

    # add theta as first column
    df.theta = thetas
    select!(df, :theta, :)

    # build rename mapping as a Vector of Pairs
    renames = [Symbol("x$i") => Symbol(string(taus[i])) for i in 1:length(taus)]

    rename!(df, renames)

    CSV.write("$name.csv", df)
end

save_grid("KK", KK)
save_grid("SE", SE)
save_grid("AP", AP)
save_grid("WE", WE)
save_grid("H", H)
save_grid("CMAX", CMAX); save_grid("CMIN", CMIN)
save_grid("TE12Mat", TE12Mat)
save_grid("TE21Mat", TE21Mat)

θ = -5.0, τ = 10.0
Initial x1 = -0.03480465422728103, x2 = 0.009810227263113383
 
θ = -4.898989898989899, τ = 10.0
Initial x1 = -0.010585968926562342, x2 = 0.056723655973465376
 
θ = -4.797979797979798, τ = 10.0
Initial x1 = 0.008848521840072607, x2 = -0.06540603453569305
 
θ = -4.696969696969697, τ = 10.0
Initial x1 = 0.0026182609181259198, x2 = 0.034196151733006996
 
θ = -4.595959595959596, τ = 10.0
Initial x1 = -0.019431168436790894, x2 = -0.07408835638088727
 
θ = -4.494949494949495, τ = 10.0
Initial x1 = -0.07378344479145023, x2 = 0.07688947700446444
 
θ = -4.393939393939394, τ = 10.0
Initial x1 = 0.035297955673049275, x2 = -0.028836600096227086
 
θ = -4.292929292929293, τ = 10.0
Initial x1 = 0.027289269509141595, x2 = 0.06522067701592049
 
θ = -4.191919191919192, τ = 10.0
Initial x1 = 0.03926296666017931, x2 = 0.02210770068483399
 
θ = -4.090909090909091, τ = 10.0
Initial x1 = -0.05642571147918796, x2 = 0.059121558524026424
 
θ = -3.98989898989899, τ = 10.0
Initial x1 = 0.0627086

"TE21Mat.csv"